# 🧠 AI Impact on Mental Growth: ML Classification

**Predict human cognitive outcomes based on AI usage patterns.**

---

### 📌 What's in this Notebook?

| Section | Description |
|---|---|
| 1. Import Libraries | Load all required Python packages |
| 2. Load & Explore Data | Understand the dataset (EDA) |
| 3. Data Preprocessing | Clean and encode the data |
| 4. Feature Importance | Find the most useful features |
| 5. Model Training | Train a Random Forest Classifier |
| 6. Model Evaluation | Accuracy, precision, recall, F1 |
| 7. Conclusion | Key takeaways |

**Target:** Predict `Outcome_Label` → *Positive Growth / Stagnant / Cognitive Decline Risk*


## 📦 Step 1: Import Libraries

We need a few Python libraries. Don't worry — each one has a simple job!

In [ ]:
# 🔢 Data handling
import pandas as pd
import numpy as np

# 📊 Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# 🤖 Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# ⚙️ Ignore minor warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")


## 📂 Step 2: Load & Explore the Dataset

Let's load our CSV file and take a first look at the data.

In [ ]:
# Load the dataset
df = pd.read_csv('/kaggle/input/ai-impact-on-human-mental-growth/AI_Impact_on_Human_Mental_Growth.csv')

print(f"📐 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


In [ ]:
# What columns do we have?
print("📋 Column Names:")
for col in df.columns:
    print(f"  • {col}")


In [ ]:
# Basic statistics for numerical columns
df.describe().T.round(2)


### 🎯 Target Variable Distribution

Let's see how many samples belong to each outcome class.

In [ ]:
# Count of each class in the target
target_counts = df['Outcome_Label'].value_counts()
print(target_counts)

# Plot
plt.figure(figsize=(8, 4))
colors = ['#4CAF50', '#FF9800', '#F44336']
target_counts.plot(kind='bar', color=colors, edgecolor='black', width=0.6)
plt.title('Distribution of Outcome Labels', fontsize=14, fontweight='bold')
plt.xlabel('Outcome Label')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### 🔍 Missing Values Check

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing = missing[missing > 0]

print("⚠️ Columns with missing values:")
print(missing)
print(f"\nTotal missing: {missing.sum():,}")


### 📊 Correlation Heatmap (Numerical Features)

In [ ]:
# Select only numeric columns for correlation
num_cols = df.select_dtypes(include='number').drop(columns=['Row_ID']).columns

plt.figure(figsize=(12, 8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.5, fmt='.1f')
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 📈 Weekly AI Usage by Outcome

In [ ]:
plt.figure(figsize=(9, 4))
for label, color in zip(df['Outcome_Label'].unique(), ['#4CAF50', '#FF9800', '#F44336']):
    subset = df[df['Outcome_Label'] == label]['Weekly_AI_Usage_Hours'].dropna()
    subset.plot(kind='kde', label=label, color=color, linewidth=2)

plt.title('Weekly AI Usage Hours by Outcome', fontsize=13, fontweight='bold')
plt.xlabel('Weekly AI Usage Hours')
plt.legend()
plt.tight_layout()
plt.show()


## 🧹 Step 3: Data Preprocessing

Before training a model, we need to:
1. **Drop useless columns** (like ID and free-text columns)
2. **Fill missing values**
3. **Encode categorical columns** (turn text → numbers, because ML models only understand numbers)


In [ ]:
# ── 1. Drop columns that won't help the model ──
drop_cols = ['Row_ID', 'Positive_Effects_of_AI', 'Negative_Effects_of_AI']
df = df.drop(columns=drop_cols)
print(f"Dropped: {drop_cols}")

# ── 2. Fill missing values ──
# AI_Tool_Category: fill blanks with 'Unknown'
df['AI_Tool_Category'] = df['AI_Tool_Category'].fillna('Unknown')
print(f"✅ Missing values handled. Remaining nulls: {df.isnull().sum().sum()}")


In [ ]:
# ── 3. Encode categorical columns to numbers ──
le = LabelEncoder()

cat_cols = [
    'Era', 'Scenario', 'Region', 'Demographic_Group',
    'Education_Level', 'Profession', 'AI_Tool_Category'
]

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Encode the target column
label_encoder = LabelEncoder()
df['Outcome_Label'] = label_encoder.fit_transform(df['Outcome_Label'])

# Show class mapping
print("🏷️ Target class mapping:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i} → {cls}")

print(f"\n✅ Encoding done! Dataset shape: {df.shape}")


## 🔬 Step 4: Feature Importance

Before training, let's quickly find out which features matter most using a lightweight Random Forest.

In [ ]:
# Split features and target
X = df.drop('Outcome_Label', axis=1)
y = df['Outcome_Label']

# Quick fit to get feature importances
quick_rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
quick_rf.fit(X, y)

# Plot top 15 features
importances = pd.Series(quick_rf.feature_importances_, index=X.columns)
top15 = importances.sort_values(ascending=True).tail(15)

plt.figure(figsize=(9, 6))
top15.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 15 Most Important Features', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


## 🤖 Step 5: Model Training

We use **Random Forest** — an ensemble of many Decision Trees. It's robust, accurate, and handles mixed data well.

We split: **80% training** → model learns | **20% testing** → we evaluate.

In [ ]:
# ── Train / Test Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42,    # reproducibility
    stratify=y          # keep class balance in both splits
)

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Testing samples  : {X_test.shape[0]:,}")


In [ ]:
# ── Train the Random Forest ──
rf_model = RandomForestClassifier(
    n_estimators=100,   # 100 decision trees
    random_state=42,
    n_jobs=-1           # use all CPU cores
)

rf_model.fit(X_train, y_train)
print("✅ Model trained successfully!")


## 📊 Step 6: Model Evaluation

Let's see how well our model performs on unseen test data.

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"🎯 Test Accuracy: {acc * 100:.2f}%")
print()

# Detailed report
target_names = label_encoder.classes_
print("📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))


### 🔲 Confusion Matrix

Shows where the model gets confused between classes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# Per-class accuracy visualization
report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
f1_scores = {cls: report[cls]['f1-score'] for cls in target_names}

plt.figure(figsize=(8, 4))
bars = plt.bar(f1_scores.keys(), f1_scores.values(), 
               color=['#4CAF50', '#FF9800', '#F44336'], edgecolor='black', width=0.5)
plt.ylim(0.9, 1.01)
plt.title('F1-Score per Class', fontsize=13, fontweight='bold')
plt.ylabel('F1-Score')
for bar, val in zip(bars, f1_scores.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.3f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()


## ✅ Step 7: Conclusion

### 🏆 Model Performance Summary

| Metric | Score |
|---|---|
| **Accuracy** | ~98.1% |
| **Precision (avg)** | ~0.98 |
| **Recall (avg)** | ~0.98 |
| **F1-Score (avg)** | ~0.98 |

---

### 🔑 Key Takeaways

1. **Random Forest** achieved **~98% accuracy** on this 3-class classification task.

2. **Top predictors** of cognitive outcome include:
   - `Mental_Wellbeing_Score`
   - `Human_Creativity_Score`
   - `Adaptability_Score`
   - `Critical_Thinking_Ability`
   - `AI_Dependency_Score`

3. **AI Tool Category** and **Weekly AI Usage Hours** also contribute meaningfully — showing that *how* someone uses AI matters for mental growth outcomes.

4. **Data quality tip:** Three columns had missing values (`AI_Tool_Category`, `Positive_Effects_of_AI`, `Negative_Effects_of_AI`). We handled them by filling with `'Unknown'` or dropping text-heavy columns not suited for tree models.

---

### 💡 What Could Be Improved?

- Try **XGBoost / LightGBM** for potentially faster training with similar accuracy
- Use **SHAP values** for deeper model explainability
- Experiment with **hyperparameter tuning** (GridSearchCV / Optuna)

---

> *This notebook is beginner-friendly and designed to be a solid starting point. If you found it useful, please upvote! ⬆️*
